This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [ ]:
testable_data = data.get_testable_data("Example\\inputs\\case study 1 input\\pain points full.csv")
codes = data.get_codes("Example\\inputs\\case study 1 input\\short titles+descriptions.csv")
all_scores = scores.get_BERT_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Consensus code" column of testable_data to all_scores_expanded
all_scores_expanded["Consensus code"] = testable_data["Consensus code"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

,Input phrase,1,2,3,4,5,6,7,8,9,10,11,Consensus code
0,cutting wood,0.254405,0.232129,0.076255,0.394942,0.144240,0.452531,0.240936,0.353553,0.334331,0.139246,0.090399,0
1,didn�t know how to use lathe,0.603036,0.382273,0.033618,0.328100,0.249983,0.541873,0.382468,0.215903,0.330588,0.153461,0.043812,1
2,Finding drill,0.315796,0.563597,0.080384,0.269875,0.251557,0.282534,0.299470,0.167122,0.633441,0.116149,0.062875,9
3,Taking out trash,0.112805,0.040450,0.122271,0.139765,0.198486,0.341823,0.166689,0.386303,0.227238,0.455778,0.120643,10
4,Finding clamp,0.291589,0.027972,0.054233,0.113307,0.487997,0.065859,0.196089,0.012188,0.043719,0.022960,0.011962,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Incorrect size gloves,0.140642,0.321357,0.560952,0.144068,0.117461,0.022232,0.090810,0.014490,0.157015,0.080610,0.086145,3
395,Uncleaned machines from previous users,0.130850,0.014225,0.056702,0.103913,0.266866,0.316293,0.226428,0.229121,0.144441,0.200273,0.148900,6
396,Machine incorrectly set up by previous user,0.246331,0.082706,0.064274,0.097088,0.216418,0.268523,0.282743,0.118411,0.149840,0.204331,0.151828,1
397,Unusable wood scarps were discarded in wrong p...,0.245419,0.155071,0.082647,0.167975,0.323915,0.393105,0.183969,0.445865,0.134988,0.327505,0.223021,8


In [34]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 11 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) >= min) 
        & (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Consensus code"].tolist()
    predictions = all_scores_expanded_filtered[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    f1s = f1_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], average=None, zero_division=0.0) 
    mtx = confusion_matrix(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11])
    kappa = cohen_kappa_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    return [mtx, f1, kappa]

In [ ]:
rows = []
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows.append({"min": i, "max": j, "kappa": results[2], "f1": results[1]})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_BERT_title+desc_painpoints.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expec

,min,max,kappa,f1
0,0.95,1.00,NaN,0.000000
1,0.90,0.95,NaN,0.090909
2,0.85,0.90,NaN,0.000000
3,0.80,0.85,1.000000,0.363636
4,0.75,0.80,1.000000,0.636364
5,0.70,0.75,1.000000,0.969697
6,0.65,0.70,0.975414,0.836180
7,0.60,0.65,0.849346,0.827413
8,0.55,0.60,0.879365,0.740107
9,0.50,0.55,0.845118,0.702098
